# 가우시안 프로세스 실습

**Gaussian Process · GP · 크리깅 · Kriging**

함수 자체에 확률분포를 두어 예측값과 불확실성을 함께 주는 모델.

소재 분야에서 이해하기: 실험이 적은 영역에서 예측 불확실성이 크게 나온다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 가우시안 프로세스 문서](https://scikit-learn.org/stable/modules/gaussian_process.html)

## 1. 예측과 불확실성을 함께

관측이 드문 구간에서 불확실성이 커지는지 확인합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel

x_obs = np.array([0.05, 0.1, 0.15, 0.2, 0.7, 0.75, 0.95])[:, None]
y_obs = np.sin(3 * np.pi * x_obs[:, 0]) + rng.normal(0, 0.05, x_obs.shape[0])
grid = np.linspace(0, 1, 300)[:, None]

kernel = ConstantKernel(1.0) * RBF(0.15) + WhiteKernel(1e-3)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0).fit(x_obs, y_obs)
mean, std = gp.predict(grid, return_std=True)

plt.plot(grid, np.sin(3 * np.pi * grid[:, 0]), 'k--', label='true')
plt.plot(grid, mean, label='GP mean')
plt.fill_between(grid[:, 0], mean - 2 * std, mean + 2 * std, alpha=0.25, label='±2σ')
plt.scatter(x_obs, y_obs, c='red', zorder=5, label='observations')
plt.legend(); plt.show()
print('관측이 없는 0.3-0.65 구간의 표준편차 평균 %.3f' % std[(grid[:, 0] > 0.3) & (grid[:, 0] < 0.65)].mean())
print('관측이 있는 0.05-0.2 구간의 표준편차 평균 %.3f' % std[(grid[:, 0] > 0.05) & (grid[:, 0] < 0.2)].mean())

## 2. 커널 길이 척도의 영향

In [ ]:
for length in (0.03, 0.15, 0.6):
    kernel = ConstantKernel(1.0) * RBF(length, length_scale_bounds='fixed') + WhiteKernel(1e-3)
    model = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0).fit(x_obs, y_obs)
    mean = model.predict(grid)
    plt.plot(grid, mean, label='length scale %.2f' % length)
plt.scatter(x_obs, y_obs, c='red', zorder=5)
plt.legend(); plt.show()
print('길이 척도가 작으면 관측 사이를 급하게 오가고, 크면 지나치게 매끄러워집니다.')

## 3. 해석

GP는 관측이 없는 곳에서 불확실성이 자동으로 커집니다. 이 성질 때문에 베이지안 최적화와
능동학습의 기본 도구로 쓰입니다. 데이터가 수천 건을 넘으면 계산 비용이 급격히 커집니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#gaussian-process)을 여세요.